# ATC mapping (Role D · Enrich)

Every drug name in the reviews gets its **ATC class** (*Anatomical Therapeutic Chemical* classification of the WHO). The validation step then checks whether drugs that patients experience alike also share a pharmacological class (research question 3), for example whether Sertraline and Lexapro, both SSRIs, have similar experience profiles.

| Step | What | Output |
|---|---|---|
| D1 | Load the clean reviews and the drug families made by A | |
| D2 | One row per drug name, with its number of reviews and main condition | |
| D3 | Look up every name in RxNorm: name → active ingredient → ATC codes | `rxnav_cache.json.gz` |
| D4 | Fill the gaps: brand ↔ generic families (A), then a short list checked by hand | |
| D5 | ATC levels and class names, coverage and checks | **`drug_atc_mapping.csv`**, `drug_atc_codes.csv`, `atc_class_names.csv` |
| D6 | One ATC class per drug for each condition in scope (birth control, depression, pain, acne) | **`drug_condition_atc.csv`** |

All numbers for the report are collected in `run_stats` and saved to `role_d_atc_run_stats.json` at the end.

## Which source, and why
The project plan left open *which source we use for the drug-to-ATC mapping*. The reviews come from Drugs.com, a US site, so most names are **US brand names** (*Nexplanon*, *Sprintec*, *Lo Loestrin Fe*) or generic names. A mapping needs two steps: brand → active ingredient, then active ingredient → ATC code.

| Source | Brand → ingredient | Ingredient → ATC | Access |
|---|---|---|---|
| WHO ATC/DDD Index | no | yes (the official one) | web pages only, no download or API |
| **RxNorm** (US National Library of Medicine), through the **RxNav API** | **yes, US brands** | **yes, the WHO codes** | free, no key |
| Wikidata, ChEMBL | few brands | yes | ingredient names only |

**Chosen: RxNorm.** One source does both steps, and its ATC codes are the WHO ones. RxNorm links an ATC code to the *active ingredient* (or the combination of ingredients), not to a product. That matters for a few products, handled in D4 and D6.

**ATC levels** (example: sertraline). The validation compares drugs at **level 4** (the pharmacological class, e.g. SSRIs) and level 3.

| Level | Code | Meaning |
|---|---|---|
| 1 | N | Nervous system |
| 2 | N06 | Psychoanaleptics |
| 3 | N06A | Antidepressants |
| 4 | N06AB | Selective serotonin reuptake inhibitors (SSRIs) |
| 5 | N06AB06 | sertraline |

## 0 · Setup and parameters
All settings live here. To experiment, change a value and run the notebook again.

*Internet*: D3 calls the RxNav API. Every answer is saved in `rxnav_cache.json.gz`, so a second run (or a run with that file as input) needs no internet. On Kaggle, turn on *Settings → Internet* for the first run.

In [1]:
import gzip
import hashlib
import html
import json
import re
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import requests
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

# where the data is read from and written to (Kaggle first, then a local copy of the repository)
INPUT_DIRS = [Path('/kaggle/input'), Path('../data-cleaning-and-lsh/outputs'), Path('../data/raw'), Path('data/raw')]
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# D3 parameters
RXNAV_URL = 'https://rxnav.nlm.nih.gov/REST'
RXNAV_CACHE_FILE = 'rxnav_cache.json.gz'   # every API answer, so the notebook can be re-run offline
RXNAV_MAX_REQUESTS_PER_SECOND = 15         # NLM asks for at most 20 per second
RXNAV_N_THREADS = 8

# D5 / D6 parameters
SCOPE_CONDITIONS = ['Birth Control', 'Depression', 'Pain', 'Acne']   # the conditions the project clusters
N_SPOT_CHECK = 25                          # random drug names printed for the check by hand
RANDOM_SEED = 42

run_stats = {}                             # every number for the report is collected here
print('outputs go to', OUTPUT_DIR)

outputs go to outputs


## D1 · Load the clean reviews and the drug families
**Normal case (Kaggle):** add A's notebook *Clean and deduplicate* as input (*Add Input → Notebook Output*); D1 reads `reviews_clean_deduplicated.parquet` and `drug_families.csv` from it.

**Fallback (local copy without A's files):** D1 reads the raw UCI files from `data/raw` and repeats the only part of A that the mapping needs, with A's own rules: the text cleaning and the exact deduplication of A2–A3 (one review listed under a brand and a generic name is one review, and the names become `drug_name_aliases`), then the drug families of A6. The near-duplicate removal of A4 is skipped: it removes 87 of about 128,750 reviews, so the review counts differ by less than 0.1%. Condition names are only freed from scraping junk here, not repaired: the four conditions in scope are not affected by A's repairs.

In [2]:
def find_input_file(file_name):
    for folder in INPUT_DIRS:
        if folder.exists():
            matches = sorted(folder.rglob(file_name))
            if matches:
                return matches[0]
    return None


# ---- fallback: the parts of A2, A3 and A6 that the mapping needs (same rules as A's notebook) ----
WHITESPACE_PATTERN = re.compile(r'\s+')
NON_ALPHANUMERIC_PATTERN = re.compile(r'[^a-z0-9\s]')
ZERO_WIDTH_SPACE = chr(0x200B)
REVIEW_DATE_FORMATS = ['%d-%b-%y', '%B %d, %Y', '%Y-%m-%d']


def clean_review_text(raw_text):
    text = html.unescape(html.unescape(str(raw_text)))
    text = text.replace(ZERO_WIDTH_SPACE, '').strip().strip('"').strip()
    return WHITESPACE_PATTERN.sub(' ', text)


def normalise_review_text(clean_text):
    text = clean_text.lower().replace("'", '')
    text = NON_ALPHANUMERIC_PATTERN.sub(' ', text)
    return WHITESPACE_PATTERN.sub(' ', text).strip()


def parse_review_dates(date_strings):
    best_parsed_dates = None
    for date_format in REVIEW_DATE_FORMATS:
        parsed_dates = pd.to_datetime(date_strings, format=date_format, errors='coerce')
        if best_parsed_dates is None or parsed_dates.notna().sum() > best_parsed_dates.notna().sum():
            best_parsed_dates = parsed_dates
    return best_parsed_dates


def load_raw_reviews():
    raw_parts = []
    for file_stem in ['drugsComTrain_raw', 'drugsComTest_raw']:
        for extension, separator in [('.csv', ','), ('.tsv', '\t')]:
            raw_file_path = find_input_file(file_stem + extension)
            if raw_file_path is not None:
                break
        else:
            raise FileNotFoundError(f'{file_stem} not found: add A\'s notebook output or the raw UCI files')
        raw_part = pd.read_csv(raw_file_path, sep=separator)
        raw_parts.append(raw_part.rename(columns={raw_part.columns[0]: 'uniqueID'}))
    raw_reviews = pd.concat(raw_parts, ignore_index=True).rename(columns={
        'uniqueID': 'review_id', 'drugName': 'drug_name', 'condition': 'condition_name',
        'review': 'review_text_raw', 'rating': 'rating', 'date': 'review_date'})
    raw_reviews['review_date'] = parse_review_dates(raw_reviews['review_date'])
    raw_reviews['rating'] = pd.to_numeric(raw_reviews['rating']).round().astype(int)
    return raw_reviews


def rebuild_deduplicated_reviews(raw_reviews):
    # A2 + A3: clean names and text, then one row per (normalised text, rating, date) with all its drug names
    reviews = raw_reviews.copy()
    reviews['drug_name'] = reviews['drug_name'].map(lambda name: html.unescape(str(name)).strip())
    condition_names = reviews['condition_name'].map(lambda name: None if pd.isna(name) else html.unescape(str(name)).strip())
    reviews['condition_name'] = condition_names.where(~condition_names.fillna('').str.contains('</span>'), None)
    normalised_text = reviews['review_text_raw'].map(clean_review_text).map(normalise_review_text)
    md5 = lambda text: hashlib.md5(text.encode('utf-8')).hexdigest()
    reviews['duplicate_key'] = (normalised_text.map(md5) + '|' + reviews['rating'].astype(str)
                                + '|' + reviews['review_date'].astype(str)).map(md5)
    reviews_by_key = reviews.groupby('duplicate_key')
    drug_name_aliases = reviews_by_key['drug_name'].agg(lambda names: sorted(set(names)))
    first_known_condition = reviews_by_key['condition_name'].agg(
        lambda names: names.dropna().iloc[0] if names.notna().any() else None)
    deduplicated = (reviews.assign(condition_is_missing=reviews['condition_name'].isna())
                           .sort_values(['condition_is_missing', 'review_id'])
                           .drop_duplicates('duplicate_key', keep='first'))
    deduplicated['drug_name_aliases'] = deduplicated['duplicate_key'].map(drug_name_aliases)
    deduplicated['condition_name'] = deduplicated['duplicate_key'].map(first_known_condition)
    return deduplicated[['review_id', 'drug_name', 'drug_name_aliases', 'condition_name', 'rating']].reset_index(drop=True)


def build_drug_families(reviews):
    # A6 step 1: names that appeared on the same review are one medicine (connected groups of names)
    alias_lists = reviews['drug_name_aliases']
    all_drug_names = sorted({name for aliases in alias_lists for name in aliases})
    name_index = {name: index for index, name in enumerate(all_drug_names)}
    links = np.array([(name_index[aliases[0]], name_index[other_name])
                      for aliases in alias_lists for other_name in aliases[1:]], dtype=np.int64).reshape(-1, 2)
    name_graph = coo_matrix((np.ones(len(links)), (links[:, 0], links[:, 1])),
                            shape=(len(all_drug_names), len(all_drug_names)))
    _, family_of_name = connected_components(name_graph, directed=False)
    reviews_per_drug_name = reviews['drug_name'].value_counts()
    return (pd.DataFrame({'drug_name': all_drug_names, 'drug_family_id': family_of_name})
            .assign(n_reviews=lambda table: table['drug_name'].map(reviews_per_drug_name).fillna(0).astype(int))
            .sort_values(['n_reviews', 'drug_name'], ascending=[False, True])
            .groupby('drug_family_id', sort=False)
            .agg(drug_family_name=('drug_name', 'first'), drug_names=('drug_name', ' | '.join),
                 n_drug_names=('drug_name', 'size'), n_reviews=('n_reviews', 'sum'))
            .sort_values('n_reviews', ascending=False))

In [3]:
clean_reviews_path = find_input_file('reviews_clean_deduplicated.parquet')
if clean_reviews_path is not None:
    reviews = pd.read_parquet(clean_reviews_path, columns=['review_id', 'drug_name', 'drug_name_aliases',
                                                          'condition_name', 'rating'])
    reviews['drug_name_aliases'] = reviews['drug_name_aliases'].map(list)
    run_stats['input'] = f'A\'s output: {clean_reviews_path.name}'
else:
    reviews = rebuild_deduplicated_reviews(load_raw_reviews())
    run_stats['input'] = 'fallback: raw UCI files with A2-A3 exact deduplication (no A4)'

drug_families_path = find_input_file('drug_families.csv')
drug_families = (pd.read_csv(drug_families_path, index_col='drug_family_id') if drug_families_path is not None
                 else build_drug_families(reviews))
family_id_of_drug_name = {name: family_id for family_id, names in drug_families['drug_names'].items()
                          for name in names.split(' | ')}

run_stats['n_reviews'] = int(len(reviews))
run_stats['n_drug_families'] = int(len(drug_families))
print(run_stats['input'])
print(f'{len(reviews):,} reviews, {len(family_id_of_drug_name):,} drug names in {len(drug_families):,} families')

fallback: raw UCI files with A2-A3 exact deduplication (no A4)
128,751 reviews, 3,671 drug names in 1,389 families


## D2 · One row per drug name
A review keeps one drug name (`drug_name`) and lists every name it appeared under (`drug_name_aliases`). The mapping is made for **every name in the aliases**, so any of them can be joined later. For each name:
- `n_reviews`: reviews whose `drug_name` is this name (the count A uses);
- `main_condition`: the most frequent condition of the reviews listed under this name. It is only shown to help the check by hand.

In [4]:
reviews_per_alias = reviews[['drug_name_aliases', 'condition_name']].explode('drug_name_aliases')
main_condition = (reviews_per_alias.dropna(subset=['condition_name'])
                  .groupby('drug_name_aliases')['condition_name'].agg(lambda names: names.value_counts().index[0]))
drug_names = (pd.DataFrame({'drug_name': sorted(family_id_of_drug_name)})
              .assign(n_reviews=lambda table: table['drug_name'].map(reviews['drug_name'].value_counts()).fillna(0).astype(int),
                      main_condition=lambda table: table['drug_name'].map(main_condition),
                      drug_family_id=lambda table: table['drug_name'].map(family_id_of_drug_name))
              .assign(drug_family_name=lambda table: table['drug_family_id'].map(drug_families['drug_family_name']))
              .sort_values(['n_reviews', 'drug_name'], ascending=[False, True])
              .reset_index(drop=True))
run_stats['n_drug_names'] = int(len(drug_names))
print(f'{len(drug_names):,} drug names; {int((drug_names["n_reviews"] > 0).sum()):,} of them are the kept name of at least one review')
drug_names.head(10)

3,671 drug names; 2,390 of them are the kept name of at least one review


,drug_name,n_reviews,main_condition,drug_family_id,drug_family_name
0,Nexplanon,2892,Birth Control,813,Nexplanon
1,Ethinyl estradiol / norethindrone,2226,Birth Control,169,Ethinyl estradiol / norethindrone
2,Sertraline,1868,Depression,267,Sertraline
3,Ethinyl estradiol / norgestimate,1861,Birth Control,797,Ethinyl estradiol / norgestimate
4,Levonorgestrel,1719,Birth Control,107,Levonorgestrel
5,Phentermine,1582,Weight Loss,90,Phentermine
6,Implanon,1506,Birth Control,813,Nexplanon
7,Gabapentin,1407,Anxiety,585,Gabapentin
8,Ethinyl estradiol / levonorgestrel,1331,Birth Control,129,Ethinyl estradiol / levonorgestrel
9,Citalopram,1307,Anxiety and Stress,478,Citalopram


## D3 · Look up every name in RxNorm
For each drug name, three questions to the RxNav API:

1. **Which RxNorm concept is this name?** Tried in this order, the first that answers wins:
   - exact match (RxNorm ignores case and word order);
   - exact match without formulation words such as *XR*, *XL*, *ER*, *ODT* (*Effexor XR* → *Effexor*), because RxNorm puts the formulation in the product, not in the brand;
   - exact match among **old names** and other vocabularies, for brands no longer sold (*Implanon*, *Belviq*), which RxNorm keeps as retired concepts;
   - *approximate* match, accepted only when **every word** of the name appears in the RxNorm name: *Sprintec* → *Sprintec KIT*, *Lo Loestrin Fe* → *Lo Loestrin Fe KIT*. Without that rule the approximate search returns unrelated products.
2. **Which active ingredient(s)?** A brand (*Zoloft*) or a pack (*Sprintec KIT*) is followed to its ingredient (*sertraline*) or, for combination products, to the **combination** (*ethinyl estradiol / norgestimate*), because ATC gives combinations their own code. A salt (*azelaic acid*, *sodium hyaluronate*) goes to its ingredient only, not to the combinations that contain it. Retired concepts keep their ingredients in their history.
3. **Which ATC codes?** RxNorm stores the WHO level-5 codes of each ingredient. An ingredient can have **several** codes, one per use: metronidazole has 5 (mouth, skin, vagina, systemic antibiotic, antiprotozoal). All are kept here; D6 picks one per condition. When the ingredient itself has no code, in this order:
   - the code of its **salt** (*levothyroxine* → *levothyroxine sodium*, H03AA01);
   - the level-4 classes NLM gives to the **products** that contain it (`ATCPROD`): copper → intrauterine contraceptives (G02BA), buprenorphine / naloxone → drugs used in opioid dependence (N07BC). These classes are only used here, as a last resort, because for single products they can be wrong (they put *Mirena* and *Plan B* with the progestogen pills, G03AC);
   - for a combination, the codes of its ingredients.

   `atc_code_basis` says which one was used.

The answers are cached: the first run makes about 15,000 requests (15–20 minutes at 15 per second), later runs none.

In [5]:
class RxNavClient:
    # RxNav REST client with a rate limit, retries and a cache of every answer (keyed by the request)
    def __init__(self, cache_path, max_requests_per_second=RXNAV_MAX_REQUESTS_PER_SECOND):
        self.cache = json.loads(gzip.decompress(cache_path.read_bytes())) if cache_path is not None else {}
        self.session = requests.Session()
        self.lock = threading.Lock()
        self.seconds_between_requests = 1 / max_requests_per_second
        self.next_request_time = 0.0
        self.n_requests = 0

    def get(self, path, **params):
        key = path + '?' + urlencode(sorted(params.items()))
        if key in self.cache:
            return self.cache[key]
        for attempt in range(5):
            with self.lock:                            # space the requests out, whatever the number of threads
                wait = self.next_request_time - time.monotonic()
                self.next_request_time = max(self.next_request_time, time.monotonic()) + self.seconds_between_requests
            if wait > 0:
                time.sleep(wait)
            try:
                response = self.session.get(f'{RXNAV_URL}/{path}', params=params, timeout=30)
            except requests.RequestException:
                response = None
            if response is not None and response.status_code == 200:
                with self.lock:
                    self.cache[key] = response.json()
                    self.n_requests += 1
                return self.cache[key]
            if response is not None and 400 <= response.status_code < 500 and response.status_code != 429:
                raise ValueError(f'RxNav refused {response.url} ({response.status_code})')
            time.sleep(2 ** attempt)                   # network error, 429 or 5xx: wait and try again
        raise RuntimeError(f'RxNav did not answer {key}')

    def save(self, path):
        with self.lock:
            path.write_bytes(gzip.compress(json.dumps(self.cache, sort_keys=True).encode('utf-8')))


WORD_PATTERN = re.compile(r'[a-z0-9]+')
FORMULATION_WORDS = {'xr', 'xl', 'er', 'sr', 'cr', 'cd', 'la', 'dr', 'xt', 'odt', 'ec', 'es', 'hp'}


def words_of(text):
    return set(WORD_PATTERN.findall(text.lower()))


def without_formulation_words(drug_name):
    kept_words = [word for word in drug_name.split() if word.lower() not in FORMULATION_WORDS]
    return ' '.join(kept_words) if kept_words else drug_name


def find_rxnorm_concepts(client, drug_name):
    # (RxNorm concept ids, how they were found): the first of the four searches that answers
    short_name = without_formulation_words(drug_name)
    searches = [(drug_name, {'search': 2}, 'rxnorm_exact'),
                (short_name, {'search': 2}, 'rxnorm_exact_without_formulation'),
                (drug_name, {'allsrc': 1}, 'rxnorm_old_or_other_name')]
    for name, options, method in searches:
        if method == 'rxnorm_exact_without_formulation' and short_name == drug_name:
            continue
        concept_ids = client.get('rxcui.json', name=name, **options).get('idGroup', {}).get('rxnormId', [])
        if concept_ids:
            return concept_ids, method
    candidates = (client.get('approximateTerm.json', term=drug_name, maxEntries=10, option=1)
                        .get('approximateGroup', {}).get('candidate', []))
    name_words = words_of(short_name)
    for candidate in sorted(candidates, key=lambda candidate: int(candidate['rank'])):
        if candidate.get('name') and name_words <= words_of(candidate['name']):
            return [candidate['rxcui']], 'rxnorm_approximate'
    return [], None


def related_concepts(client, rxcui, term_types):
    concept_groups = client.get(f'rxcui/{rxcui}/related.json', tty=term_types).get('relatedGroup', {}).get('conceptGroup', [])
    return {group['tty']: group.get('conceptProperties', []) for group in concept_groups}


def find_ingredients(client, rxcui):
    # ({rxcui: (name, tty)} of the ingredient (IN) or combination (MIN) behind a concept, the concept's (name, tty))
    properties = client.get(f'rxcui/{rxcui}/properties.json').get('properties') or {}
    if properties.get('tty') in ('IN', 'MIN'):
        return {rxcui: (properties['name'], properties['tty'])}, (properties['name'], properties['tty'])
    if properties.get('tty') == 'PIN':                            # a salt (sertraline hydrochloride) -> its ingredient
        chosen = related_concepts(client, rxcui, 'IN').get('IN', [])
        if chosen:
            return ({concept['rxcui']: (concept['name'], concept['tty']) for concept in chosen},
                    (properties['name'], properties['tty']))
    elif properties:
        related = related_concepts(client, rxcui, 'IN MIN')
        chosen = related.get('MIN') or related.get('IN') or []   # a combination product -> its combination
        if chosen:
            return ({concept['rxcui']: (concept['name'], concept['tty']) for concept in chosen},
                    (properties['name'], properties['tty']))
    # retired concept, or one without relations: its history still names its ingredients
    history = client.get(f'rxcui/{rxcui}/historystatus.json').get('rxcuiStatusHistory') or {}
    attributes = history.get('attributes') or {}
    ingredient_list = (history.get('derivedConcepts') or {}).get('ingredientConcept') or []
    ingredients = {item['ingredientRxcui']: (item['ingredientName'], 'IN') for item in ingredient_list}
    if len(ingredients) > 1:                                     # RxNorm names a combination 'a / b' (alphabetical)
        combination_name = ' / '.join(sorted(name for name, _ in ingredients.values()))
        for combination_id in client.get('rxcui.json', name=combination_name, search=2).get('idGroup', {}).get('rxnormId', []):
            combination = client.get(f'rxcui/{combination_id}/properties.json').get('properties') or {}
            if combination.get('tty') == 'MIN':
                ingredients = {combination_id: (combination['name'], 'MIN')}
                break
    return ingredients, (attributes.get('name', ''), attributes.get('tty', ''))


def atc_codes_of(client, rxcui):
    answer = client.get(f'rxcui/{rxcui}/property.json', propName='ATC').get('propConceptGroup') or {}
    return {item['propValue'] for item in answer.get('propConcept', [])}


def find_atc_codes(client, rxcui, term_type):
    # (codes, basis) of one ingredient or combination, with the fallbacks described above
    codes = atc_codes_of(client, rxcui)
    if codes:
        return codes, 'ingredient'
    if term_type == 'IN':
        for salt in related_concepts(client, rxcui, 'PIN').get('PIN', []):
            codes |= atc_codes_of(client, salt['rxcui'])
        if codes:
            return codes, 'salt of the ingredient'
    product_classes = (client.get('rxclass/class/byRxcui.json', rxcui=rxcui, relaSource='ATCPROD')
                             .get('rxclassDrugInfoList', {}).get('rxclassDrugInfo', []))
    codes = {item['rxclassMinConceptItem']['classId'] for item in product_classes}
    if codes:
        return codes, 'products (level 4 only)'
    if term_type == 'MIN':
        for ingredient in related_concepts(client, rxcui, 'IN').get('IN', []):
            codes |= find_atc_codes(client, ingredient['rxcui'], 'IN')[0]
        if codes:
            return codes, 'ingredients of the combination'
    return set(), None


def map_drug_name_with_rxnorm(client, drug_name):
    concept_ids, match_method = find_rxnorm_concepts(client, drug_name)
    result = {'drug_name': drug_name, 'match_method': match_method}
    if not concept_ids:
        return result
    ingredients, matched_concepts = {}, []
    for rxcui in concept_ids:
        concept_ingredients, matched_concept = find_ingredients(client, rxcui)
        ingredients.update(concept_ingredients)
        matched_concepts.append(matched_concept)
    atc_codes, bases = set(), set()
    for rxcui, (_, term_type) in ingredients.items():
        codes, basis = find_atc_codes(client, rxcui, term_type)
        atc_codes |= codes
        bases.add(basis) if basis else None
    result.update({
        'rxnorm_concept': ' | '.join(sorted({name for name, _ in matched_concepts if name})),
        'rxnorm_term_type': ' | '.join(sorted({term_type for _, term_type in matched_concepts if term_type})),
        'ingredient_rxcui': ' | '.join(sorted(ingredients)),
        'generic_name': ' + '.join(sorted({name for name, _ in ingredients.values()})) or None,
        'atc_codes': sorted(atc_codes),
        'atc_code_basis': ' | '.join(sorted(bases)) or None,
    })
    return result

In [6]:
cache_input_path = find_input_file(RXNAV_CACHE_FILE) or (OUTPUT_DIR / RXNAV_CACHE_FILE if (OUTPUT_DIR / RXNAV_CACHE_FILE).exists() else None)
rxnav_client = RxNavClient(cache_input_path)
print('cache:', cache_input_path, f'({len(rxnav_client.cache):,} answers)')

started = time.perf_counter()
try:
    with ThreadPoolExecutor(max_workers=RXNAV_N_THREADS) as pool:
        rxnorm_rows = list(pool.map(lambda name: map_drug_name_with_rxnorm(rxnav_client, name), drug_names['drug_name']))
finally:
    rxnav_client.save(OUTPUT_DIR / RXNAV_CACHE_FILE)          # keep what was fetched, also after an error
seconds_rxnav = time.perf_counter() - started

rxnav_version = rxnav_client.get('version.json')               # cached: the RxNorm release the mapping was made with
rxnorm_mapping = pd.DataFrame(rxnorm_rows)
rxnorm_mapping['atc_codes'] = rxnorm_mapping['atc_codes'].map(lambda codes: codes if isinstance(codes, list) else [])
has_codes = rxnorm_mapping['atc_codes'].str.len() > 0
run_stats['rxnorm_lookup'] = {
    'rxnorm_release': rxnav_version.get('version'), 'rxnav_api_version': rxnav_version.get('apiVersion'),
    'n_new_requests': rxnav_client.n_requests, 'seconds': round(seconds_rxnav, 1),
    'n_names_by_match_method': {str(method): int(count) for method, count in
                                rxnorm_mapping['match_method'].fillna('no match').value_counts().items()},
    'n_names_matched_without_atc_code': int((rxnorm_mapping['match_method'].notna() & ~has_codes).sum()),
    'n_names_by_atc_code_basis': {str(basis): int(count) for basis, count in
                                  rxnorm_mapping.loc[has_codes, 'atc_code_basis'].value_counts().items()},
}
print(f'RxNorm release {rxnav_version.get("version")}, {rxnav_client.n_requests:,} new requests, {seconds_rxnav:.0f} s')
print(json.dumps(run_stats['rxnorm_lookup'], indent=1))

cache: outputs/rxnav_cache.json.gz (15,089 answers)


RxNorm release 02-Nov-2026, 0 new requests, 0 s
{
 "rxnorm_release": "02-Nov-2026",
 "rxnav_api_version": "3.1.357",
 "n_new_requests": 0,
 "seconds": 0.3,
 "n_names_by_match_method": {
  "rxnorm_exact": 2512,
  "rxnorm_old_or_other_name": 832,
  "rxnorm_approximate": 159,
  "no match": 109,
  "rxnorm_exact_without_formulation": 59
 },
 "n_names_matched_without_atc_code": 489,
 "n_names_by_atc_code_basis": {
  "ingredient": 2396,
  "products (level 4 only)": 554,
  "salt of the ingredient": 89,
  "ingredients of the combination": 33,
  "ingredient | products (level 4 only)": 1
 }
}


## D4 · Fill the gaps
Two sources fill what RxNorm leaves open, in this order of priority (a later step overrides an earlier one):

1. **Brand ↔ generic pairs (A3).** Drugs.com lists one review under a brand and its generic name, and A3 kept all the names of a review in `drug_name_aliases`. A name without an ATC code takes the codes of the mapped name it **shares the most reviews with**: *Mononessa* gets the codes of *Ethinyl estradiol / norgestimate*. Only names listed on the same review are used, not A's whole families: a few families chain different medicines through one review listed under both (*Sertraline* and *Lorazepam*), and D5 lists them.
2. **A short list checked by hand** (`MANUAL_ATC_CODES`), with codes from the WHO ATC index, for products where an ingredient-level code is wrong or missing, limited to names with many reviews or in the conditions in scope:
   - **IUDs**: *Mirena* is levonorgestrel, but the WHO classifies the device as an intrauterine contraceptive (G02BA03), not as a progestogen pill (G03AC03). The copper IUD gets its level-5 code (G02BA02) instead of the level-4 class of D3.
   - **Emergency contraception** (*Plan B* and the other 1.5 mg levonorgestrel pills): G03AD01, not the progestogen code.
   - **Topical diclofenac** (*Voltaren Gel*, *Flector Patch*, *Pennsaid*), reviewed for pain: M02AA15, not the tablets (M01AB05), which D6 would otherwise pick for pain.

Each name keeps where its codes came from in `atc_source`.

In [7]:
# Product-level corrections, codes from the WHO ATC index (https://atcddd.fhi.no/atc_ddd_index/)
LEVONORGESTREL_IUD = (['G02BA03'], 'levonorgestrel IUD: plastic IUD with progestogen (G02BA03), not systemic progestogen')
COPPER_IUD = (['G02BA02'], 'copper IUD: plastic IUD with copper (G02BA02); RxNorm has no level-5 code for copper')
EMERGENCY_PILL = (['G03AD01'], 'levonorgestrel 1.5 mg emergency contraceptive pill (G03AD01)')
TOPICAL_DICLOFENAC = (['M02AA15'], 'diclofenac gel or patch: topical NSAID (M02AA15), not the tablets (M01AB05)')
MANUAL_ATC_CODES = {
    'Mirena': LEVONORGESTREL_IUD, 'Skyla': LEVONORGESTREL_IUD, 'Kyleena': LEVONORGESTREL_IUD,
    'Liletta': LEVONORGESTREL_IUD,
    'Copper': COPPER_IUD, 'ParaGard': COPPER_IUD,
    'Plan B': EMERGENCY_PILL, 'Plan B One-Step': EMERGENCY_PILL, 'Next Choice': EMERGENCY_PILL,
    'My Way': EMERGENCY_PILL, 'Take Action': EMERGENCY_PILL,
    'Opcicon One-Step': EMERGENCY_PILL, 'Aftera': EMERGENCY_PILL, 'EContra EZ': EMERGENCY_PILL,
    'Fallback Solo': EMERGENCY_PILL,
    'Voltaren Gel': TOPICAL_DICLOFENAC, 'Flector Patch': TOPICAL_DICLOFENAC, 'Pennsaid': TOPICAL_DICLOFENAC,
}
MANUAL_GENERIC_NAMES = {'Copper': 'copper', 'ParaGard': 'copper'}   # names RxNorm cannot give a generic for

In [8]:
drug_atc = drug_names.merge(rxnorm_mapping, on='drug_name', how='left')
drug_atc['atc_codes'] = drug_atc['atc_codes'].map(lambda codes: codes if isinstance(codes, list) else [])
has_rxnorm_codes = drug_atc['atc_codes'].str.len() > 0
drug_atc['atc_source'] = drug_atc['match_method'].where(has_rxnorm_codes)
drug_atc['atc_source_name'] = drug_atc['drug_name'].where(has_rxnorm_codes)
drug_atc['note'] = None

# 1 · brand <-> generic pairs: names listed on the same review (A3). An unmapped name takes the codes of the
#     mapped name it shares the most reviews with (ties: the one with more reviews)
pair_counts = Counter((name, partner) for aliases in reviews['drug_name_aliases'] if len(aliases) > 1
                      for name in aliases for partner in aliases if partner != name)
names_listed_together = pd.DataFrame([(name, partner, count) for (name, partner), count in pair_counts.items()],
                                     columns=['drug_name', 'partner_name', 'n_shared_reviews'])
mapped_by_rxnorm = drug_atc[has_rxnorm_codes].set_index('drug_name')
partner_of_name = (names_listed_together[~names_listed_together['drug_name'].isin(mapped_by_rxnorm.index)
                                         & names_listed_together['partner_name'].isin(mapped_by_rxnorm.index)]
                   .assign(partner_reviews=lambda table: table['partner_name'].map(mapped_by_rxnorm['n_reviews']))
                   .sort_values(['n_shared_reviews', 'partner_reviews', 'partner_name'], ascending=[False, False, True])
                   .drop_duplicates('drug_name').set_index('drug_name')['partner_name'])
is_filled_by_pair = drug_atc['drug_name'].isin(partner_of_name.index)
partner_name = drug_atc['drug_name'].map(partner_of_name)
partner_value = lambda column: partner_name.map(mapped_by_rxnorm[column])
drug_atc['atc_codes'] = drug_atc['atc_codes'].where(~is_filled_by_pair, partner_value('atc_codes'))
drug_atc['atc_code_basis'] = drug_atc['atc_code_basis'].where(~is_filled_by_pair, partner_value('atc_code_basis'))
drug_atc['generic_name'] = drug_atc['generic_name'].where(drug_atc['generic_name'].notna() | ~is_filled_by_pair,
                                                          partner_value('generic_name'))
drug_atc['atc_source'] = drug_atc['atc_source'].where(~is_filled_by_pair, 'brand_generic_pair')
drug_atc['atc_source_name'] = drug_atc['atc_source_name'].where(~is_filled_by_pair, partner_name)

# 2 · the list checked by hand (overrides everything else)
missing_manual_names = sorted(set(MANUAL_ATC_CODES) - set(drug_atc['drug_name']))
assert not missing_manual_names, f'names in MANUAL_ATC_CODES that are not in the data: {missing_manual_names}'
is_manual = drug_atc['drug_name'].isin(MANUAL_ATC_CODES)
manual_entry = drug_atc['drug_name'].map(lambda name: MANUAL_ATC_CODES.get(name, (None, None)))
drug_atc['atc_codes'] = drug_atc['atc_codes'].where(~is_manual, manual_entry.str[0])
drug_atc['note'] = drug_atc['note'].where(~is_manual, manual_entry.str[1])
drug_atc['atc_code_basis'] = drug_atc['atc_code_basis'].where(~is_manual, 'WHO index, checked by hand')
drug_atc['atc_source'] = drug_atc['atc_source'].where(~is_manual, 'manual')
drug_atc['atc_source_name'] = drug_atc['atc_source_name'].where(~is_manual, None)
drug_atc['generic_name'] = drug_atc['generic_name'].fillna(drug_atc['drug_name'].map(MANUAL_GENERIC_NAMES))

is_filled_by_pair = is_filled_by_pair & ~is_manual
run_stats['gap_filling'] = {
    'n_names_filled_by_brand_generic_pair': int(is_filled_by_pair.sum()),
    'n_reviews_filled_by_brand_generic_pair': int(drug_atc.loc[is_filled_by_pair, 'n_reviews'].sum()),
    'n_names_set_by_hand': int(is_manual.sum()),
    'n_reviews_set_by_hand': int(drug_atc.loc[is_manual, 'n_reviews'].sum()),
}
print(json.dumps(run_stats['gap_filling'], indent=1))
print('\nnames filled from a brand <-> generic pair, most reviewed first (check that the partner is the same medicine):')
drug_atc.loc[is_filled_by_pair, ['drug_name', 'n_reviews', 'main_condition', 'atc_source_name', 'atc_codes']].head(25)

{
 "n_names_filled_by_brand_generic_pair": 470,
 "n_reviews_filled_by_brand_generic_pair": 3237,
 "n_names_set_by_hand": 18,
 "n_reviews_set_by_hand": 4225
}

names filled from a brand <-> generic pair, most reviewed first (check that the partner is the same medicine):


,drug_name,n_reviews,main_condition,atc_source_name,atc_codes
105,Seasonique,281,Birth Control,Ethinyl estradiol / levonorgestrel,"[G03AA07, G03AA08, G03AB03]"
120,Polyethylene glycol 3350 with electrolytes,261,Bowel Preparation,MoviPrep,[A06AD]
151,Minastrin 24 Fe,203,Birth Control,Ethinyl estradiol / norethindrone,"[G03AA05, G03AB04]"
162,Mononessa,182,Birth Control,Ethinyl estradiol / norgestimate,"[G03AA11, G03AB09]"
171,Gianvi,176,Birth Control,Drospirenone / ethinyl estradiol,[G03AA12]
178,Bactrim DS,171,Urinary Tract Infection,Sulfamethoxazole / trimethoprim,[J01EE01]
180,Orsythia,169,Birth Control,Ethinyl estradiol / levonorgestrel,"[G03AA07, G03AA08, G03AB03]"
248,LoSeasonique,108,Birth Control,Ethinyl estradiol / levonorgestrel,"[G03AA07, G03AA08, G03AB03]"
305,Jolivette,84,Birth Control,Norethindrone,"[G03AC01, G03DC02]"
309,Ortho Micronor,82,Birth Control,Norethindrone,"[G03AC01, G03DC02]"


## D5 · ATC levels, class names and checks
A level-5 code contains all the levels above it: N06AB06 → N, N06, N06A, N06AB. The class names of levels 1–4 come from RxClass (the NLM's class browser, same WHO source). A drug with several codes can belong to several level-4 classes; all of them are listed, separated by `|`.

Checks:
1. **Coverage**: share of names and, more important, share of **reviews** whose drug has an ATC code, overall and for the four conditions in scope. Unmapped drugs are left out of the validation, as the project plan says.
2. **Families that disagree**: A's families whose names RxNorm mapped to classes with nothing in common, i.e. families that join two different medicines because one review was listed under both. D4 does not use families, so the mapping is not affected; the list is **for A**, because A6 counts each family as one medicine.
3. **Unmapped names** with the most reviews: what the validation loses.
4. **Check by hand**: a random sample of names with their condition and class.

In [9]:
atc_class_list = rxnav_client.get('rxclass/allClasses.json', classTypes='ATC1-4')['rxclassMinConceptList']['rxclassMinConcept']
atc_class_names = pd.Series({item['classId']: item['className'] for item in atc_class_list if item['classId'] != '0'},
                            name='atc_class_name').rename_axis('atc_class_code').sort_index()
rxnav_client.save(OUTPUT_DIR / RXNAV_CACHE_FILE)
ATC_LEVEL_CODE_LENGTHS = {1: 1, 2: 3, 3: 4, 4: 5, 5: 7}


def atc_level_code(atc_code, level):
    return atc_code[:ATC_LEVEL_CODE_LENGTHS[level]] if len(atc_code) >= ATC_LEVEL_CODE_LENGTHS[level] else None


def unique_in_order(values):
    return list(dict.fromkeys(value for value in values if value))


# long table: one row per (drug name, ATC code)
drug_atc_codes = drug_atc[['drug_name', 'atc_codes']].explode('atc_codes').dropna().rename(columns={'atc_codes': 'atc_code'})
for level in [1, 2, 3, 4]:
    level_codes = drug_atc_codes['atc_code'].map(lambda code: atc_level_code(code, level))
    drug_atc_codes[f'atc_level_{level}'] = level_codes
    drug_atc_codes[f'atc_level_{level}_name'] = level_codes.map(atc_class_names)
drug_atc_codes = drug_atc_codes.reset_index(drop=True)

# wide table: one row per drug name, the classes of each level joined with ' | '
for level in [1, 2, 3, 4]:
    drug_atc[f'atc_level_{level}'] = drug_atc['atc_codes'].map(
        lambda codes: unique_in_order(atc_level_code(code, level) for code in sorted(codes)))
drug_atc['atc_level_4_name'] = drug_atc['atc_level_4'].map(lambda codes: [atc_class_names.get(code, code) for code in codes])
drug_atc['n_atc_level_4'] = drug_atc['atc_level_4'].str.len()
drug_atc['has_atc'] = drug_atc['n_atc_level_4'] > 0
print(f'{len(atc_class_names):,} ATC classes (levels 1-4); {len(drug_atc_codes):,} (drug name, code) rows')
print('level-4 codes without a name:', sorted(set(drug_atc_codes['atc_level_4'].dropna()) - set(atc_class_names.index)))
print('codes shorter than level 4 (product classes of level 3):', sorted(drug_atc_codes.loc[drug_atc_codes['atc_code'].str.len() < 5, 'atc_code'].unique()))

1,318 ATC classes (levels 1-4); 5,544 (drug name, code) rows
level-4 codes without a name: []
codes shorter than level 4 (product classes of level 3): ['R05X']


In [10]:
# 1 · coverage, by names and by reviews
reviews_with_atc = reviews['drug_name'].map(drug_atc.set_index('drug_name')['has_atc']).fillna(False).astype(bool)
coverage_by_source = (drug_atc.assign(atc_source=drug_atc['atc_source'].fillna('not mapped'))
                      .groupby('atc_source').agg(n_names=('drug_name', 'size'), n_reviews=('n_reviews', 'sum'))
                      .sort_values('n_reviews', ascending=False))
coverage_by_source['share_of_reviews'] = (coverage_by_source['n_reviews'] / coverage_by_source['n_reviews'].sum()).round(4)
scope_reviews = reviews[reviews['condition_name'].isin(SCOPE_CONDITIONS)]
coverage_by_condition = (pd.DataFrame({'condition_name': scope_reviews['condition_name'],
                                       'has_atc': reviews_with_atc[scope_reviews.index]})
                         .groupby('condition_name').agg(n_reviews=('has_atc', 'size'), share_of_reviews_with_atc=('has_atc', 'mean'))
                         .reindex(SCOPE_CONDITIONS).round(4))
run_stats['coverage'] = {
    'n_names_with_atc': int(drug_atc['has_atc'].sum()),
    'share_of_names_with_atc': round(float(drug_atc['has_atc'].mean()), 4),
    'share_of_reviews_with_atc': round(float(reviews_with_atc.mean()), 4),
    'n_names_with_several_level_4_classes': int((drug_atc['n_atc_level_4'] > 1).sum()),
    'by_source': {source: {'n_names': int(row['n_names']), 'n_reviews': int(row['n_reviews'])}
                  for source, row in coverage_by_source.iterrows()},
    'by_scope_condition': {condition: {'n_reviews': int(row['n_reviews']),
                                       'share_of_reviews_with_atc': float(row['share_of_reviews_with_atc'])}
                           for condition, row in coverage_by_condition.iterrows()},
}
print(f"names with an ATC code: {run_stats['coverage']['n_names_with_atc']:,} of {len(drug_atc):,} "
      f"({run_stats['coverage']['share_of_names_with_atc']:.1%}); reviews: {run_stats['coverage']['share_of_reviews_with_atc']:.1%}")
print(f"names with more than one level-4 class: {run_stats['coverage']['n_names_with_several_level_4_classes']:,}\n")
display(coverage_by_source)
coverage_by_condition

names with an ATC code: 3,547 of 3,671 (96.6%); reviews: 99.8%
names with more than one level-4 class: 915



,n_names,n_reviews,share_of_reviews
atc_source,,,
rxnorm_exact,2462,111175,0.8635
rxnorm_approximate,158,4534,0.0352
rxnorm_old_or_other_name,380,4348,0.0338
manual,18,4225,0.0328
brand_generic_pair,470,3237,0.0251
rxnorm_exact_without_formulation,59,920,0.0071
not mapped,124,312,0.0024


,n_reviews,share_of_reviews_with_atc
condition_name,,
Birth Control,19526,0.9997
Depression,7112,1.0000
Pain,5194,1.0000
Acne,4553,1.0000


In [11]:
# 2 · families whose RxNorm-mapped names have no level-4 class in common
rxnorm_mapped = drug_atc[drug_atc['atc_source'].fillna('').str.startswith('rxnorm')]
disagreeing_families = []
for family_id, members in rxnorm_mapped.groupby('drug_family_id'):
    if len(members) < 2:
        continue
    class_sets = [set(classes) for classes in members['atc_level_4']]
    if not set.intersection(*class_sets):
        disagreeing_families.append({
            'drug_family_name': members['drug_family_name'].iloc[0],
            'names_and_classes': ' ; '.join(f"{name} ({'|'.join(classes)})"
                                            for name, classes in zip(members['drug_name'], members['atc_level_4'])),
            'n_reviews': int(members['n_reviews'].sum()),
        })
disagreeing_families = pd.DataFrame(disagreeing_families).sort_values('n_reviews', ascending=False) if disagreeing_families else pd.DataFrame()
run_stats['n_families_with_disagreeing_classes'] = int(len(disagreeing_families))
run_stats['families_with_disagreeing_classes'] = disagreeing_families['drug_family_name'].tolist() if len(disagreeing_families) else []
print(f'{len(disagreeing_families)} families whose mapped names share no level-4 class')
disagreeing_families.head(15)

9 families whose mapped names share no level-4 class


,drug_family_name,names_and_classes,n_reviews
7,Duloxetine,Duloxetine (N06AX) ; Pregabalin (N02BF) ; Latuda (N05AE) ; Lurasidone (N05AE) ; Cymbalta (N06AX) ; Lyrica (N02BF),2553
6,Gabapentin,Gabapentin (N02BF) ; Tramadol (N02AX) ; Ultram (N02AX) ; Ultram ER (N02AX) ; Ultram ODT (N02AX) ; Ryzolt (N02AX) ; G...,2550
3,Sertraline,Sertraline (N06AB) ; Lorazepam (N05BA) ; Ativan (N05BA) ; Zoloft (N06AB),2460
8,Quetiapine,Quetiapine (N05AH) ; Seroquel (N05AH) ; Seroquel XR (N05AH) ; Luvox (N06AB) ; Fluvoxamine (N06AB),1217
0,Acetaminophen / hydrocodone,Acetaminophen / hydrocodone (N02AJ) ; Triazolam (N05CD) ; Anexsia (N02AJ) ; Lorcet (N02AJ) ; Maxidone (N02AJ) ; Halc...,695
1,Angeliq,Angeliq (G03AA) ; Drospirenone / estradiol (G03FA),20
2,Octreotide,Octreotide (H01CB) ; Insulin regular (A10AB|A10AC|A10AD|A10AE|A10AF) ; Insulin isophane / insulin regular (A10AD) ; ...,19
5,Ocular lubricant,Lacrisert (S01XA) ; GenTeal (S01KA) ; Hypotears (S01XA) ; Tears Again (S01XA) ; TheraTears (S01XA) ; FreshKote (S01X...,17
4,Menthol / zinc oxide,Menthol / zinc oxide (C05AX|D02AB) ; Calmoseptine (D02AC),4


In [12]:
# 3 · unmapped names with the most reviews
unmapped = drug_atc[~drug_atc['has_atc']]
run_stats['unmapped'] = {'n_names': int(len(unmapped)), 'n_reviews': int(unmapped['n_reviews'].sum()),
                         'top_names': unmapped['drug_name'].head(15).tolist()}
print(f"{len(unmapped):,} names without an ATC code, {unmapped['n_reviews'].sum():,} reviews")
unmapped[['drug_name', 'n_reviews', 'main_condition', 'match_method', 'rxnorm_concept', 'generic_name']].head(20)

124 names without an ATC code, 312 reviews


,drug_name,n_reviews,main_condition,match_method,rxnorm_concept,generic_name
449,Multivitamin,48,Dietary Supplementation,rxnorm_old_or_other_name,MULTIVITAMINS,NaN
693,Emollients,21,Dry Skin,rxnorm_old_or_other_name,EMOLLIENTS,NaN
722,Dermal filler,19,acial Wrinkles,None,NaN,NaN
828,EpiCeram,14,Dry Skin,rxnorm_old_or_other_name,EPICERAM,NaN
903,Saliva substitutes,12,Xerostomia,rxnorm_old_or_other_name,ARTIFICIAL SALIVA,NaN
939,Systane,11,Dry Eye Disease,rxnorm_old_or_other_name,PROPRIETARY,NaN
954,Echinacea,10,Cold Symptoms,rxnorm_exact,Echinacea preparation,Echinacea preparation
985,Barium sulfate,9,Computed Tomography,rxnorm_exact,barium sulfate,barium sulfate
993,EnteraGam,9,Irritable Bowel Syndrome,rxnorm_old_or_other_name,PROPRIETARY,NaN
1029,Bioflavonoids,8,Osteoarthritis,rxnorm_old_or_other_name,BIOFLAVONOIDS | Bioflavonoid drugs for capillary stabilizing | CITRUS BIOFLAVONOIDS,NaN


In [13]:
# 4 · check by hand: random names with at least one review
spot_check = (drug_atc[drug_atc['has_atc'] & (drug_atc['n_reviews'] > 0)]
              .sample(N_SPOT_CHECK, random_state=RANDOM_SEED).sort_values('n_reviews', ascending=False))
spot_check[['drug_name', 'n_reviews', 'main_condition', 'generic_name', 'atc_source', 'atc_codes', 'atc_level_4_name']]

,drug_name,n_reviews,main_condition,generic_name,atc_source,atc_codes,atc_level_4_name
44,Vyvanse,601,ADHD,lisdexamfetamine,rxnorm_exact,[N06BA12],[Centrally acting sympathomimetics]
208,Bystolic,139,High Blood Pressure,nebivolol,rxnorm_exact,[C07AB12],"[Beta blocking agents, selective]"
282,Claravis,94,Acne,isotretinoin,rxnorm_exact,"[D10AD04, D10BA01]","[Retinoids for topical use in acne, Retinoids for treatment of acne]"
472,Sucralfate,45,GERD,sucralfate,rxnorm_exact,[A02BX02],[Other drugs for peptic ulcer and gastro-oesophageal reflux disease (GORD)]
483,Chlorzoxazone,42,Muscle Spasm,chlorzoxazone,rxnorm_exact,[M03BB03],"[Oxazol, thiazine, and triazine derivatives]"
508,Embeda,39,Chronic Pain,morphine / naltrexone,rxnorm_old_or_other_name,"[N02AA01, N07BB04]","[Natural opium alkaloids, Drugs used in alcohol dependence]"
510,Frovatriptan,39,Migraine,frovatriptan,rxnorm_exact,[N02CC07],[Selective serotonin (5HT1) agonists]
621,Cambia,25,Migraine,diclofenac,rxnorm_exact,"[D11AX18, M01AB05, M02AA15, S01BC03]","[Other dermatologicals, Acetic acid derivatives and related substances, Antiinflammatory preparations, non-steroids ..."
822,Allegra-D 24 Hour,14,Allergic Rhinitis,fexofenadine / pseudoephedrine,brand_generic_pair,[R01BA],[Sympathomimetics]
935,Rhofade,11,Rosacea,oxymetazoline,rxnorm_exact,"[D11AX27, R01AA05, R01AB07, S01GA04]","[Other dermatologicals, Sympathomimetics, plain, Sympathomimetics, combinations excl. corticosteroids, Sympathomimet..."


## D6 · One ATC class per review and per drug (birth control, depression, pain, acne)
The validation compares drugs **within a condition**, and many analyses (agreement between clusters and classes, colours in a figure) need **one** class per drug. D5 keeps all codes of a drug, and some drugs have codes in several level-4 classes because ATC classifies by use: clindamycin is an anti-acne preparation (D10AF01), a gynaecological anti-infective (G01AA10) and a systemic antibiotic (J01FF01).

**Which name of the review?** After A3 a review keeps one name, and it is often the generic one: most *Mirena* reviews (an IUD) are kept as *Levonorgestrel*, whose codes are those of the progestogen pill and the emergency pill. The brand is still in `drug_name_aliases`. So each review is classified by its **most specific name**: the one with the fewest level-4 classes (ties: the kept name). It is stored as `atc_drug_name`; for most reviews it is the kept name.

**Which class?** A drug with one level-4 class keeps it. A drug with several gets the class that fits the condition, from `PREFERRED_ATC_GROUPS`: the first group of the list that one of its codes belongs to (ties: the first code in alphabetical order). The groups follow the ATC tree, for example acne → anti-acne preparations for systemic use (D10B) first, because the isotretinoin that patients review is the oral one, then tetracycline antibiotics (J01A), because minocycline was only sold as tablets in 2008–2017 (the topical foam came in 2019), then topical anti-acne preparations (D10A). Each row keeps the rule it used in `class_choice`.

Two tables: `drug_condition_atc` (one row per condition and drug, with `n_reviews` to drop rare drugs) and `scope_review_atc` (one row per review in scope, to join with the clusters by `review_id`).

In [14]:
PREFERRED_ATC_GROUPS = {
    'Birth Control': ['G03A',                      # hormonal contraceptives for systemic use
                      'G02B'],                     # intravaginal contraceptives (IUDs, vaginal ring)
    'Depression': ['N06A',                         # antidepressants
                   'N05A', 'N03A', 'N05B', 'N06B', 'N06C'],   # antipsychotics, antiepileptics, anxiolytics, stimulants, combinations
    'Pain': ['N02A', 'N02B',                       # opioids, other analgesics
             'M01A', 'M02A',                       # anti-inflammatory drugs, systemic and topical
             'N03A', 'N06A', 'M03B', 'N01B', 'N02C'],   # antiepileptics, antidepressants, muscle relaxants, local anaesthetics, migraine
    'Acne': ['D10B',                               # anti-acne preparations for systemic use (oral isotretinoin)
             'J01A',                               # tetracyclines: minocycline was only sold as tablets until 2019
             'D10A', 'J01F',                       # topical anti-acne preparations, macrolides / lincosamides
             'G03A', 'G03H', 'C03D', 'D06A', 'D11A'],   # pills, antiandrogens, spironolactone, topical antibiotics, other skin
}
assert set(PREFERRED_ATC_GROUPS) == set(SCOPE_CONDITIONS)


def choose_atc_code(atc_codes, condition_name):
    # (code, rule): the only level-4 class, else the first preferred group of the condition, else the first code
    codes = sorted(atc_codes)
    if not codes:
        return None, 'no ATC code'
    if len({atc_level_code(code, 4) for code in codes}) == 1:
        return codes[0], 'only one class'
    for group in PREFERRED_ATC_GROUPS[condition_name]:
        matching = [code for code in codes if code.startswith(group)]
        if matching:
            return matching[0], f'preferred group {group}'
    return codes[0], 'first code (no preferred group)'


n_classes_of_name = drug_atc.set_index('drug_name')['n_atc_level_4']


def most_specific_name(kept_name, aliases):
    # the name of the review with the fewest level-4 classes (unmapped names never win); ties: the kept name
    candidates = [name for name in aliases if n_classes_of_name.get(name, 0) > 0] or [kept_name]
    fewest = min(n_classes_of_name.get(name, 0) for name in candidates)
    best = [name for name in candidates if n_classes_of_name.get(name, 0) == fewest]
    return kept_name if kept_name in best else sorted(best)[0]


scope_review_atc = reviews.loc[reviews['condition_name'].isin(SCOPE_CONDITIONS),
                               ['review_id', 'condition_name', 'drug_name', 'drug_name_aliases']].copy()
scope_review_atc['atc_drug_name'] = [most_specific_name(name, aliases) for name, aliases
                                     in zip(scope_review_atc['drug_name'], scope_review_atc['drug_name_aliases'])]
is_renamed = scope_review_atc['atc_drug_name'] != scope_review_atc['drug_name']
run_stats['n_scope_reviews_classified_by_another_name'] = int(is_renamed.sum())
print(f'{is_renamed.sum():,} of {len(scope_review_atc):,} reviews in scope are classified by a more specific name than the kept one:')
print((scope_review_atc.loc[is_renamed, 'drug_name'] + ' -> ' + scope_review_atc.loc[is_renamed, 'atc_drug_name'])
      .value_counts().head(10).to_string(), '\n')

scope_counts = (scope_review_atc.groupby(['condition_name', 'atc_drug_name']).size().rename('n_reviews').reset_index())
drug_condition_atc = scope_counts.merge(
    drug_atc[['drug_name', 'generic_name', 'drug_family_name', 'atc_codes', 'atc_source']]
    .rename(columns={'drug_name': 'atc_drug_name'}), on='atc_drug_name', how='left')
drug_condition_atc['atc_codes'] = drug_condition_atc['atc_codes'].map(lambda codes: codes if isinstance(codes, list) else [])
chosen = [choose_atc_code(codes, condition) for codes, condition in zip(drug_condition_atc['atc_codes'], drug_condition_atc['condition_name'])]
drug_condition_atc['atc_code'] = [code for code, _ in chosen]
drug_condition_atc['class_choice'] = [rule for _, rule in chosen]
for level in [4, 3, 2, 1]:
    level_codes = drug_condition_atc['atc_code'].map(lambda code: atc_level_code(code, level) if isinstance(code, str) else None)
    drug_condition_atc[f'atc_level_{level}'] = level_codes
    drug_condition_atc[f'atc_level_{level}_name'] = level_codes.map(atc_class_names)
drug_condition_atc['all_atc_codes'] = drug_condition_atc['atc_codes'].map(lambda codes: ' | '.join(sorted(codes)))
drug_condition_atc = (drug_condition_atc.drop(columns='atc_codes')
                      .sort_values(['condition_name', 'n_reviews', 'atc_drug_name'], ascending=[True, False, True])
                      .reset_index(drop=True))
scope_review_atc = (scope_review_atc.drop(columns='drug_name_aliases')    # class names are in drug_condition_atc
                    .merge(drug_condition_atc[['condition_name', 'atc_drug_name', 'generic_name', 'atc_code',
                                               'atc_level_4', 'atc_level_3']],
                           on=['condition_name', 'atc_drug_name'], how='left'))
assert scope_review_atc['review_id'].is_unique

choice_summary = drug_condition_atc.groupby(['condition_name', 'class_choice'])['n_reviews'].agg(['size', 'sum'])
choice_summary.columns = ['n_drug_names', 'n_reviews']
run_stats['scope_classes'] = {}
for condition in SCOPE_CONDITIONS:
    rows = drug_condition_atc[drug_condition_atc['condition_name'] == condition]
    with_class = rows[rows['atc_code'].notna()]
    run_stats['scope_classes'][condition] = {
        'n_drug_names': int(len(rows)), 'n_drug_names_with_class': int(len(with_class)),
        'share_of_reviews_with_class': round(float(with_class['n_reviews'].sum() / rows['n_reviews'].sum()), 4),
        'n_level_4_classes': int(with_class['atc_level_4'].nunique()),
        'n_drug_names_by_choice': {rule: int(count) for rule, count in rows['class_choice'].value_counts().items()},
    }
choice_summary

1,221 of 36,385 reviews in scope are classified by a more specific name than the kept one:
Levonorgestrel -> Mirena     1015
Levonorgestrel -> Kyleena     205
Lanacane -> Benzocaine          1 



n_drug_names  n_reviews
condition_name class_choice                                            
Acne           first code (no preferred group)             4          5
               only one class                             43       2029
               preferred group D10A                        7        545
               preferred group D10B                        6        682
               preferred group G03A                        7        439
               preferred group J01A                       10        853
Birth Control  no ATC code                                 1          5
               only one class                             31      10482
               preferred group G03A                       56       9039
Depression     first code (no preferred group)             3        100
               only one class                             89       6997
               preferred group N06A                        2         15
Pain           first code (no preferred group)             5          6
               only one class                            101       4309
               preferred group M01A                       25        499
               preferred group M02A                        2         19
               preferred group N01B                        2         93
               preferred group N02A                        4        260
               preferred group N02B                        2          8

### Check by eye: the most-reviewed drugs of each condition and their class
These are the drugs the validation will compare, so every class below should look right to someone who knows the drugs.

In [15]:
for condition in SCOPE_CONDITIONS:
    rows = drug_condition_atc[drug_condition_atc['condition_name'] == condition]
    print(f'=== {condition}: {len(rows)} drug names, {rows["n_reviews"].sum():,} reviews ===')
    print(rows[['atc_drug_name', 'n_reviews', 'generic_name', 'atc_code', 'atc_level_4_name', 'class_choice']]
          .head(15).to_string(index=False), '\n')

=== Birth Control: 88 drug names, 19,526 reviews ===
                     atc_drug_name  n_reviews                       generic_name atc_code                               atc_level_4_name         class_choice
                         Nexplanon       2883                       etonogestrel  G03AC08                                   Progestogens       only one class
 Ethinyl estradiol / norethindrone       1669  ethinyl estradiol / norethindrone  G03AA05 Progestogens and estrogens, fixed combinations preferred group G03A
                          Implanon       1496                       etonogestrel  G03AC08                                   Progestogens       only one class
  Ethinyl estradiol / norgestimate       1327   ethinyl estradiol / norgestimate  G03AA11 Progestogens and estrogens, fixed combinations preferred group G03A
                            Mirena       1320                     levonorgestrel  G02BA03                    Intrauterine contraceptives       only one class

In [16]:
# classes per condition, by reviews: how many drugs and reviews each level-4 class has
class_composition = (drug_condition_atc.dropna(subset=['atc_level_4'])
                     .groupby(['condition_name', 'atc_level_4', 'atc_level_4_name'])
                     .agg(n_drug_names=('atc_drug_name', 'size'), n_reviews=('n_reviews', 'sum'))
                     .reset_index().sort_values(['condition_name', 'n_reviews'], ascending=[True, False]))
class_composition.groupby('condition_name').head(6).reset_index(drop=True)

,condition_name,atc_level_4,atc_level_4_name,n_drug_names,n_reviews
0,Acne,D10AD,Retinoids for topical use in acne,6,1194
1,Acne,J01AA,Tetracyclines,10,853
2,Acne,G03AA,"Progestogens and estrogens, fixed combinations",15,730
3,Acne,D10BA,Retinoids for treatment of acne,6,682
4,Acne,D10AF,Antiinfectives for treatment of acne,10,475
5,Acne,C03DA,Aldosterone antagonists,2,238
6,Birth Control,G03AA,"Progestogens and estrogens, fixed combinations",62,9484
7,Birth Control,G03AC,Progestogens,15,5497
8,Birth Control,G02BA,Intrauterine contraceptives,6,3671
9,Birth Control,G02BB,Intravaginal contraceptives,2,829


## Save the tables
| File | One row per | For |
|---|---|---|
| `drug_atc_mapping.csv` | drug name (all names of the data, brands and generics) | joining ATC to the reviews: `reviews.merge(mapping, on='drug_name')` |
| `drug_atc_codes.csv` | (drug name, ATC code) with every level and its name | analyses that keep all codes of a drug |
| `drug_condition_atc.csv` | (condition in scope, drug name) with **one** class | the validation: drug profiles vs ATC class |
| `scope_review_atc.csv` | review in scope (`review_id`) with the name it is classified by and **one** class | joining ATC to the clusters, which are per review |
| `atc_class_names.csv` | ATC class (levels 1-4) | names for tables and figures |
| `rxnav_cache.json.gz` | API request | running the notebook again without internet |

| `drug_atc_mapping.csv` column | Meaning |
|---|---|
| `drug_name`, `n_reviews`, `main_condition` | name as in the reviews, reviews kept under it, its most frequent condition |
| `drug_family_id`, `drug_family_name` | A's brand ↔ generic family |
| `generic_name` | RxNorm active ingredient, or combination (`ethinyl estradiol / norgestimate`): **one name per medicine** |
| `atc_codes` | all level-5 codes, separated by ` \| ` (empty: not mapped, left out of the validation) |
| `atc_level_1` … `atc_level_4`, `atc_level_4_name`, `n_atc_level_4` | the classes of those codes |
| `atc_source`, `atc_source_name` | how the name was found (`rxnorm_exact`, `rxnorm_exact_without_formulation`, `rxnorm_old_or_other_name`, `rxnorm_approximate`), or `brand_generic_pair` (codes taken from the name in `atc_source_name`), or `manual` |
| `atc_code_basis` | where the codes come from: the `ingredient`, its `salt`, its `products (level 4 only)`, the `ingredients of the combination`, or the WHO index by hand |
| `rxnorm_concept`, `rxnorm_term_type`, `ingredient_rxcui` | what RxNorm matched (term types: BN brand, IN ingredient, MIN combination, ...) |
| `note` | reason of a manual code |

In [17]:
join_list = lambda values: ' | '.join(values)
mapping_columns = ['drug_name', 'n_reviews', 'main_condition', 'drug_family_id', 'drug_family_name', 'generic_name',
                   'atc_codes', 'atc_level_1', 'atc_level_2', 'atc_level_3', 'atc_level_4', 'atc_level_4_name', 'n_atc_level_4',
                   'atc_source', 'atc_source_name', 'atc_code_basis', 'rxnorm_concept', 'rxnorm_term_type',
                   'ingredient_rxcui', 'note']
drug_atc_mapping = drug_atc[mapping_columns].copy()
for column in ['atc_codes', 'atc_level_1', 'atc_level_2', 'atc_level_3', 'atc_level_4', 'atc_level_4_name']:
    drug_atc_mapping[column] = drug_atc_mapping[column].map(join_list)

drug_atc_mapping.to_csv(OUTPUT_DIR / 'drug_atc_mapping.csv', index=False)
drug_atc_codes.to_csv(OUTPUT_DIR / 'drug_atc_codes.csv', index=False)
drug_condition_atc.to_csv(OUTPUT_DIR / 'drug_condition_atc.csv', index=False)
scope_review_atc.to_csv(OUTPUT_DIR / 'scope_review_atc.csv', index=False)
atc_class_names.to_csv(OUTPUT_DIR / 'atc_class_names.csv')
print(drug_atc_mapping.shape, drug_atc_codes.shape, drug_condition_atc.shape, scope_review_atc.shape, atc_class_names.shape)

(3671, 20) (5544, 10) (400, 17) (36385, 8) (1318,)


## Summary for the report
Printed from this run, so it always matches the outputs above.

In [18]:
coverage = run_stats['coverage']
print(f"Source: RxNorm release {run_stats['rxnorm_lookup']['rxnorm_release']} (NLM, RxNav API), WHO ATC codes.")
print(f"1. {run_stats['n_drug_names']:,} drug names; {coverage['n_names_with_atc']:,} ({coverage['share_of_names_with_atc']:.0%}) "
      f"have an ATC code, covering {coverage['share_of_reviews_with_atc']:.1%} of the {run_stats['n_reviews']:,} reviews.")
for source, numbers in coverage['by_source'].items():
    print(f"   {source:20s} {numbers['n_names']:5,} names {numbers['n_reviews']:7,} reviews")
print(f"2. {coverage['n_names_with_several_level_4_classes']:,} names have codes in several level-4 classes; "
      f"D6 gives one class per drug and condition.")
print('3. Conditions in scope (share of reviews whose drug has a class, number of level-4 classes):')
for condition, numbers in run_stats['scope_classes'].items():
    print(f"   {condition:14s} {numbers['share_of_reviews_with_class']:.1%} of reviews, {numbers['n_drug_names']} drug names, "
          f"{numbers['n_level_4_classes']} classes")
print(f"4. {run_stats['n_scope_reviews_classified_by_another_name']:,} reviews in scope are classified by a more specific "
      f"name than the one A kept (mostly Mirena and Kyleena kept as Levonorgestrel).")
print(f"5. {run_stats['n_families_with_disagreeing_classes']} of A's families link names with no class in common "
      f"(one review listed under two medicines): {', '.join(run_stats['families_with_disagreeing_classes'][:6])}.")

Source: RxNorm release 02-Nov-2026 (NLM, RxNav API), WHO ATC codes.
1. 3,671 drug names; 3,547 (97%) have an ATC code, covering 99.8% of the 128,751 reviews.
   rxnorm_exact         2,462 names 111,175 reviews
   rxnorm_approximate     158 names   4,534 reviews
   rxnorm_old_or_other_name   380 names   4,348 reviews
   manual                  18 names   4,225 reviews
   brand_generic_pair     470 names   3,237 reviews
   rxnorm_exact_without_formulation    59 names     920 reviews
   not mapped             124 names     312 reviews
2. 915 names have codes in several level-4 classes; D6 gives one class per drug and condition.
3. Conditions in scope (share of reviews whose drug has a class, number of level-4 classes):
   Birth Control  100.0% of reviews, 88 drug names, 6 classes
   Depression     100.0% of reviews, 94 drug names, 16 classes
   Pain           100.0% of reviews, 141 drug names, 31 classes
   Acne           100.0% of reviews, 77 drug names, 17 classes
4. 1,221 reviews in sc

## Save all statistics for the report

In [19]:
(OUTPUT_DIR / 'role_d_atc_run_stats.json').write_text(json.dumps(run_stats, indent=2))
print(json.dumps(run_stats, indent=1))
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))

{
 "input": "fallback: raw UCI files with A2-A3 exact deduplication (no A4)",
 "n_reviews": 128751,
 "n_drug_families": 1389,
 "n_drug_names": 3671,
 "rxnorm_lookup": {
  "rxnorm_release": "02-Nov-2026",
  "rxnav_api_version": "3.1.357",
  "n_new_requests": 0,
  "seconds": 0.3,
  "n_names_by_match_method": {
   "rxnorm_exact": 2512,
   "rxnorm_old_or_other_name": 832,
   "rxnorm_approximate": 159,
   "no match": 109,
   "rxnorm_exact_without_formulation": 59
  },
  "n_names_matched_without_atc_code": 489,
  "n_names_by_atc_code_basis": {
   "ingredient": 2396,
   "products (level 4 only)": 554,
   "salt of the ingredient": 89,
   "ingredients of the combination": 33,
   "ingredient | products (level 4 only)": 1
  }
 },
 "gap_filling": {
  "n_names_filled_by_brand_generic_pair": 470,
  "n_reviews_filled_by_brand_generic_pair": 3237,
  "n_names_set_by_hand": 18,
  "n_reviews_set_by_hand": 4225
 },
 "coverage": {
  "n_names_with_atc": 3547,
  "share_of_names_with_atc": 0.9662,
  "share_of